# Fine-tune จับทรงกล่องจากไดไลน์ (Colab GPU)

**เป้า:** แก้ tuck-family (1/2/4/11) ที่ CLIP-kNN ตัน ~43% โดย fine-tune EfficientNet

**⚠️ ข้อควรรู้:**
- labels เป็น **proxy (engineer)** ยังมี noise → นี่คือ **pilot baseline** ไม่ใช่ผลสุดท้าย
- ใช้ **group-split กัน leakage** (revision งานเดียวกันไม่ข้าม train/val)
- เทียบกับ **CLIP-kNN baseline = 43%**

**ขั้นตอน:** Runtime → Change runtime type → GPU (T4) แล้ว Run all → อัป `dieline_train.zip` ตอน cell อัปโหลด


In [ ]:
!pip -q install timm scikit-learn


### 1. อัปโหลด dieline_train.zip (จาก bench/ ในเครื่อง)


In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()          # เลือก dieline_train.zip
zipfile.ZipFile(list(up.keys())[0]).extractall('data')
print('รูป:', len(os.listdir('data/img')))


### 2. โหลด labels + group-split (กัน leakage)


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
df = pd.read_csv('data/labels.csv'); df['y'] = df['label'] - 1
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, va = next(gss.split(df, groups=df['group_id']))
train_df, val_df = df.iloc[tr].reset_index(drop=True), df.iloc[va].reset_index(drop=True)
leak = set(train_df.group_id) & set(val_df.group_id)
print('train', len(train_df), '| val', len(val_df), '| group leakage:', len(leak))
assert len(leak)==0, 'มี leakage!'
print('train ต่อทรง:', train_df.label.value_counts().sort_index().to_dict())


### 3. Dataset + transforms


In [ ]:
import torch, timm, numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
SZ=320
train_tf = T.Compose([T.Resize((SZ,SZ)), T.RandomRotation(4), T.ColorJitter(0.1,0.1), T.ToTensor(), T.Normalize([0.5]*3,[0.5]*3)])
val_tf   = T.Compose([T.Resize((SZ,SZ)), T.ToTensor(), T.Normalize([0.5]*3,[0.5]*3)])
class DS(Dataset):
    def __init__(s, d, tf): s.d=d; s.tf=tf
    def __len__(s): return len(s.d)
    def __getitem__(s, i):
        r=s.d.iloc[i]; im=Image.open('data/img/'+r.file).convert('RGB')
        return s.tf(im), int(r.y)
tl = DataLoader(DS(train_df,train_tf), batch_size=32, shuffle=True, num_workers=2)
vl = DataLoader(DS(val_df,val_tf), batch_size=64, num_workers=2)


### 4. เทรน (EfficientNet-B0 pretrained, class-weighted, 15 epoch)


In [ ]:
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=12).to(dev)
cnt = train_df.y.value_counts().sort_index()
w = torch.tensor([len(train_df)/(12*cnt.get(i,1)) for i in range(12)], dtype=torch.float).to(dev)
crit = nn.CrossEntropyLoss(weight=w)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=15)
for ep in range(15):
    model.train(); tot=0
    for x,y in tl:
        x,y=x.to(dev),y.to(dev); opt.zero_grad()
        loss=crit(model(x),y); loss.backward(); opt.step(); tot+=loss.item()
    sched.step()
    model.eval(); c=n=0
    with torch.no_grad():
        for x,y in vl:
            c+=(model(x.to(dev)).argmax(1).cpu()==y).sum().item(); n+=len(y)
    print('ep%2d  loss %.3f  val_acc %.1f%%' % (ep+1, tot/len(tl), 100*c/n))


### 5. ประเมิน (เทียบ CLIP-kNN 43%)


In [ ]:
from sklearn.metrics import classification_report
model.eval(); P=[]; Y=[]
with torch.no_grad():
    for x,y in vl:
        P += (model(x.to(dev)).argmax(1).cpu()+1).tolist(); Y += (y+1).tolist()
P, Y = np.array(P), np.array(Y)
print('12-way accuracy: %.0f%%   (CLIP-kNN baseline = 43%%)' % (100*(P==Y).mean()))
tk = np.isin(Y,[1,2,4,11])
print('tuck-family (1/2/4/11): %.0f%%' % (100*(P[tk]==Y[tk]).mean()))
print('custom-vs-standard   : %.0f%%' % (100*((P==12)==(Y==12)).mean()))
print()
print(classification_report(Y, P, zero_division=0))


### 6. เซฟโมเดล


In [ ]:
torch.save(model.state_dict(), 'dieline_effb0.pt')
from google.colab import files; files.download('dieline_effb0.pt')
